# 📝 Exercise M6.01

The aim of this notebook is to investigate if we can tune the hyperparameters
of a bagging regressor and evaluate the gain obtained.

We will load the California housing dataset and split it into a training and a
testing set.

In [1]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

data, target = fetch_california_housing(as_frame=True, return_X_y=True)
target *= 100  # rescale the target in k$
data_train, data_test, target_train, target_test = train_test_split(
    data, target, random_state=0, test_size=0.5
)

<div class="admonition note alert alert-info">
<p class="first admonition-title" style="font-weight: bold;">Note</p>
<p class="last">If you want a deeper overview regarding this dataset, you can refer to the
Appendix - Datasets description section at the end of this MOOC.</p>
</div>

Create a `BaggingRegressor` and provide a `DecisionTreeRegressor` to its
parameter `estimator`. Train the regressor and evaluate its generalization
performance on the testing set using the mean absolute error.

In [2]:
# Write your code here.
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.metrics import mean_absolute_error

bagged_tree = BaggingRegressor(
    estimator=DecisionTreeRegressor(max_depth=3),
    n_estimators=100
)
_ = bagged_tree.fit(data_train, target_train)
r2 = bagged_tree.score(data_test, target_test)
print(f"r2: {r2}")

target_test_prediction = bagged_tree.predict(data_test)
mae = mean_absolute_error(target_test, target_test_prediction)
print(f"Mean absolute error: {mae:.2f} k$")

r2: 0.5534906866826959
Mean absolute error: 57.61 k$


Now, create a `RandomizedSearchCV` instance using the previous model and tune
the important parameters of the bagging regressor. Find the best parameters
and check if you are able to find a set of parameters that improve the default
regressor still using the mean absolute error as a metric.

<div class="admonition tip alert alert-warning">
<p class="first admonition-title" style="font-weight: bold;">Tip</p>
<p class="last">You can list the bagging regressor's parameters using the <tt class="docutils literal">get_params</tt> method.</p>
</div>

In [5]:
# Write your code here.
bagged_tree.get_params()

{'bootstrap': True,
 'bootstrap_features': False,
 'estimator__ccp_alpha': 0.0,
 'estimator__criterion': 'squared_error',
 'estimator__max_depth': 3,
 'estimator__max_features': None,
 'estimator__max_leaf_nodes': None,
 'estimator__min_impurity_decrease': 0.0,
 'estimator__min_samples_leaf': 1,
 'estimator__min_samples_split': 2,
 'estimator__min_weight_fraction_leaf': 0.0,
 'estimator__monotonic_cst': None,
 'estimator__random_state': None,
 'estimator__splitter': 'best',
 'estimator': DecisionTreeRegressor(max_depth=3),
 'max_features': 1.0,
 'max_samples': None,
 'n_estimators': 100,
 'n_jobs': None,
 'oob_score': False,
 'random_state': None,
 'verbose': 0,
 'warm_start': False}

In [6]:
from scipy.stats import randint
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    "n_estimators": randint(1, 100),
    "max_samples": [0.5, 0.8, 1.0],
    "max_features": [0.5, 0.8, 1.0],
    "estimator__max_depth": randint(3, 10)
}

search = RandomizedSearchCV(
    bagged_tree, param_grid, n_iter=50, scoring="neg_mean_absolute_error"
)
_ = search.fit(data_train, target_train)

In [7]:
import pandas as pd

columns = [f"param_{name}" for name in param_grid.keys()]
columns += ["mean_test_error", "std_test_error"]
cv_results = pd.DataFrame(search.cv_results_)
cv_results["mean_test_error"] = -cv_results["mean_test_score"]
cv_results["std_test_error"] = cv_results["std_test_score"]
cv_results[columns].sort_values(by="mean_test_error")

,param_n_estimators,param_max_samples,param_max_features,param_estimator__max_depth,mean_test_error,std_test_error
9,75,1.0,0.8,9,38.250722,0.786842
25,78,0.8,0.8,9,38.360653,1.410418
36,56,1.0,1.0,9,38.800753,1.299733
30,47,0.8,0.8,9,38.881255,0.804318
11,63,0.8,1.0,9,39.072681,1.007855
41,24,0.8,0.8,8,40.774945,0.483802
21,58,0.5,1.0,8,40.896841,1.043133
2,9,0.8,0.8,8,40.957336,1.117801
8,91,0.8,0.8,7,42.041689,1.255215
34,30,0.5,1.0,7,42.690936,0.983285


In [8]:
target_predicted = search.predict(data_test)
print(f"Mean absolute error: {mean_absolute_error(target_test, target_predicted):.2f} k$")

Mean absolute error: 37.76 k$
